In [1]:
import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv("data/clean/reviews_clean.csv", parse_dates=["at"])

# Negative reviews only, split into the two periods we compare
neg = df[df["score"] <= 2].copy()
neg["period"] = None
neg.loc[neg["at"] < "2024-09-01", "period"] = "Before Keeta"
neg.loc[(neg["at"] >= "2024-09-01") & (neg["at"] < "2025-08-01"), "period"] = "Keeta Saudi only"
neg = neg.dropna(subset=["period"])

# Test: did the share of price complaints really change for each competitor?
results = []
for app in ["HungerStation", "Jahez", "Mrsool", "ToYou"]:
    data = neg[neg["app"] == app]
    table = pd.crosstab(data["period"], data["Price"])
    chi2, p, dof, expected = chi2_contingency(table)
    before = data[data["period"] == "Before Keeta"]["Price"].mean() * 100
    after = data[data["period"] == "Keeta Saudi only"]["Price"].mean() * 100
    results.append({
        "app": app,
        "before_%": round(before, 1),
        "after_%": round(after, 1),
        "p_value": round(p, 4),
        "significant": p < 0.05,
    })

pd.DataFrame(results)

,app,before_%,after_%,p_value,significant
0,HungerStation,16.6,19.3,0.0159,True
1,Jahez,12.6,14.4,0.2914,False
2,Mrsool,11.0,14.8,0.0004,True
3,ToYou,17.8,18.2,0.8621,False


In [2]:
# Label ALL reviews with the 3 periods
df["period"] = "3) Keeta Gulf"
df.loc[df["at"] < "2025-08-01", "period"] = "2) Keeta Saudi only"
df.loc[df["at"] < "2024-09-01", "period"] = "1) Before Keeta"
df["is_negative"] = df["score"] <= 2

def compare(app, a, b):
    data = df[(df["app"] == app) & (df["period"].isin([a, b]))]
    table = pd.crosstab(data["period"], data["is_negative"])
    chi2, p, dof, expected = chi2_contingency(table)
    pct = data.groupby("period")["is_negative"].mean() * 100
    return {"app": app, "from": a, "to": b,
            "negative_from_%": round(pct[a], 1),
            "negative_to_%": round(pct[b], 1),
            "p_value": round(p, 4),
            "significant": p < 0.05}

tests = [
    compare("Keeta",         "2) Keeta Saudi only", "3) Keeta Gulf"),
    compare("HungerStation", "1) Before Keeta",     "2) Keeta Saudi only"),
    compare("HungerStation", "2) Keeta Saudi only", "3) Keeta Gulf"),
    compare("Jahez",         "2) Keeta Saudi only", "3) Keeta Gulf"),
]
pd.DataFrame(tests)

,app,from,to,negative_from_%,negative_to_%,p_value,significant
0,Keeta,2) Keeta Saudi only,3) Keeta Gulf,29.7,57.6,0.0,True
1,HungerStation,1) Before Keeta,2) Keeta Saudi only,54.9,63.2,0.0,True
2,HungerStation,2) Keeta Saudi only,3) Keeta Gulf,63.2,51.3,0.0,True
3,Jahez,2) Keeta Saudi only,3) Keeta Gulf,62.7,72.1,0.0,True
